# SPINE-GPE v7 — Dossiê de Reprodutibilidade da Fase 0 v1.0.0

Este notebook valida a cadeia certificada da Fase 0 e cria um dossiê portátil, com manifestos, ambiente, scripts, relatórios, hashes, ZIP, lock e freeze. Nenhuma estimativa científica é recalculada.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
from pathlib import Path
import shutil, zipfile, os

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
PACKAGE_ZIP = Path("/content/SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_PACKAGE_v1.0.0.zip")
INSTALL_DIR = ROOT / "scripts" / "phase0_reproducibility_dossier_v100"
EXPECTED_MASTER_LOCK_SHA256 = "38d3e5e5380032fd61997a8e913113d5430f8670af9c51870a47edcc97232c53"
EXPECTED_MASTER_FREEZE_SHA256 = "3ee7e7eca7f71d6e4216621e30bb71541829fa5709be13d2a4ffb46155312454"
RAW_HASH_MODE = "all"  # arquivístico; use "declared" apenas para um teste rápido
MAX_EMBED_MB = 256

assert PACKAGE_ZIP.is_file(), f"Envie o pacote para {PACKAGE_ZIP}"
if INSTALL_DIR.exists():
    shutil.rmtree(INSTALL_DIR)
INSTALL_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as zf:
    zf.extractall(INSTALL_DIR)
print("Instalado em:", INSTALL_DIR)
print("Arquivos:", len(list(INSTALL_DIR.rglob("*"))))


In [ ]:
import subprocess, sys

ENGINE = INSTALL_DIR / "SPINE_GPEv7_PHASE0_REPRODUCIBILITY_DOSSIER_v1.0.0.py"
assert ENGINE.is_file(), ENGINE

cmd_audit = [
    sys.executable, str(ENGINE),
    "--root", str(ROOT),
    "--mode", "audit",
    "--run-id", "phase0_reproducibility_audit_v100",
    "--expected-master-lock-sha256", EXPECTED_MASTER_LOCK_SHA256,
    "--expected-master-freeze-sha256", EXPECTED_MASTER_FREEZE_SHA256,
    "--raw-hash-mode", "declared",
    "--strict",
]
print("Executando audit:")
print(" ".join(cmd_audit))
print()
proc = subprocess.Popen(cmd_audit, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
audit_exit = proc.wait()
print("\nAudit exit code:", audit_exit)
assert audit_exit == 0


## Build arquivístico

Com `RAW_HASH_MODE="all"`, esta etapa calcula SHA-256 de todos os arquivos em `01_raw`. Ela pode demorar, mas não copia os dados brutos para o ZIP.

In [ ]:
RUN_ID = "phase0_reproducibility_final_v100"
cmd_build = [
    sys.executable, str(ENGINE),
    "--root", str(ROOT),
    "--mode", "build",
    "--run-id", RUN_ID,
    "--expected-master-lock-sha256", EXPECTED_MASTER_LOCK_SHA256,
    "--expected-master-freeze-sha256", EXPECTED_MASTER_FREEZE_SHA256,
    "--raw-hash-mode", RAW_HASH_MODE,
    "--max-embed-mb", str(MAX_EMBED_MB),
    "--strict",
]
print("Executando build:")
print(" ".join(cmd_build))
print()
proc = subprocess.Popen(cmd_build, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
build_exit = proc.wait()
print("\nBuild exit code:", build_exit)
assert build_exit == 0


In [ ]:
import hashlib, json

def sha256_file(path, chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

LOCK = ROOT / "00_admin" / "SPINE_GPE_PHASE0_REPRODUCIBILITY_DOSSIER_LOCK.json"
FREEZE = ROOT / "00_admin" / "SPINE_GPE_PHASE0_REPRODUCIBILITY_DOSSIER_FREEZE.json"
assert LOCK.is_file(), LOCK
assert FREEZE.is_file(), FREEZE
lock=json.loads(LOCK.read_text(encoding="utf-8"))
freeze=json.loads(FREEZE.read_text(encoding="utf-8"))
zip_path=Path(lock["dossier_zip"])
assert lock["status"] == "REPRODUCIBILITY_DOSSIER_CERTIFIED"
assert lock["critical_failures"] == []
assert freeze["status"] == "FROZEN"
assert zip_path.is_file()
assert sha256_file(zip_path) == lock["dossier_zip_sha256"]
print("Dossier status:", lock["status"])
print("Freeze status:", freeze["status"])
print("ZIP:", zip_path)
print("ZIP SHA-256:", lock["dossier_zip_sha256"])
print("Next phase:", lock.get("next_phase"))
print("\nPHASE 0 REPRODUCIBILITY DOSSIER CERTIFIED AND FROZEN")


In [ ]:
# Autoverificação do ZIP extraído em diretório temporário
import tempfile
with tempfile.TemporaryDirectory() as td:
    td=Path(td)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(td)
    verify = td / "verify_dossier.py"
    result = subprocess.run([sys.executable, str(verify), "--dossier-root", str(td)], text=True, capture_output=True)
    print(result.stdout)
    assert result.returncode == 0, result.stderr
